# Chat completions with LumiOpen/Llama-Poro-2-70B-Instruct 

Now we are ready to test chat-tuned [`LumiOpen/Llama-Poro-2-70B-Instruct`](https://huggingface.co/LumiOpen/Llama-Poro-2-70B-Instruct). 

## 0. Get an access token and configure the OpenAI client

Aitta's API is **OpenAI-compatible**, so we use the standard [`openai`](https://pypi.org/project/openai/) Python library, which has already been installed in this workspace. No Aitta-specific library or setup code is needed: we only point the client to Aitta's base URL (`https://aitta-api.csc.fi/openai/v1`) and use an Aitta access token instead of an OpenAI API key. The requests go to Aitta and the model runs on the LUMI supercomputer.

**Creating an access token**
1. Go to [aitta-auth.csc.fi/myToken](https://aitta-auth.csc.fi/myToken), or click the `Generate token` link on the [Aitta homepage](https://aitta.csc.fi). 
2. Log in with your Haka, MyAccessId/Puhuri or CSC account.
3. Select one of your LUMI projects. Your access token will be tied to this project. If you have only one LUMI project, you may not see the corresponding screen.
4. The next page displays your access token. You can copy it by clicking on the small clipboard icon.

Access tokens are currently valid for 90 days. The validity ends earlier if the LUMI project reaches its end date or your user account is closed.

In [1]:
# Set genereated access token here  
access_token = "<obtain your access token from https://aitta.csc.fi/ and enter it here>"

In [9]:
import openai

# Configure Client instance with API URL and access token
client = openai.OpenAI(
    api_key=access_token,
    base_url="https://aitta-api.csc.fi/openai/v1"
)

model = "LumiOpen/Llama-Poro-2-70B-Instruct"

# 1. Create chat completions

You must be eager to get started with actually getting responses from model based on your prompt.
Let's create one using  `client.chat.completions.create()` method and then explore how to fine-tune its parameters.

If the model is not already running, Aitta first allocates resources on LUMI and loads the model, so **the first response can take several minutes**. Following requests are answered much faster.

Here again diagram of the first API call using client for chat completions: 

![api-call-diagram-for-chat-completions](./images/API-chat-completions-bold.png)

In [42]:
# perform chat completion with the OpenAI client
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain with two sentences what is the LUMI supercomputer."
        }
    ],
    model=model
)

print(chat_completion.choices[0].message.content)

The LUMI (Large Unified Modern Infrastructure) supercomputer is a world-leading high-performance computing (HPC) system located in Kajaani, Finland, designed to provide extensive computational and artificial intelligence resources for scientific research and innovation. It is one of the most powerful supercomputers in the world, offering outstanding performance for a variety of applications, from climate modeling and biosciences to artificial intelligence and materials science.


# 2. Messages and roles

We just used the `client.chat.completions.create` method to generate a chat completion. It requires two parameters: `model` and `messages`.

* **`model`** is the id of the model you want to use, here `LumiOpen/Llama-Poro-2-70B-Instruct`.
* **`messages`** is a list that holds the conversation. Each message is a dictionary with a `role` (who is "speaking") and `content` (the text of the message). 

There are three roles used for the `messages`. Only `user` is required:

| Role | Purpose | Example |
|----|----|---|
| `system` | Optional. Instructions that set the model's behaviour for the whole conversation, e.g. its tone, style or task. Usually the first message, and optional. | `{"role": "system", "content": "You are an assistant who always gives concise answers."}` |
| `user` | Your input: a question, a prompt or a task. | `{"role": "user", "content": "How does AI work?"}` |
| `assistant` | Optional. The model's replies. In code, you add them to the list yourself, either to continue a conversation (the model's earlier replies) or to show the model example answers that guide its style and format (few-shot prompting). | `{"role": "assistant", "content": "AI works by learning patterns from large amounts of data."}` |

**In some cases, a single `user` message is enough.** Only the `user` role is required: the simplest request is a list with one `user` message, as in the first example of this notebook. The `system` and `assistant` roles are optional tools for when you need more control. Use `system` to set the model's behaviour, and `assistant` to continue a conversation or to give example answers.


In [ ]:
messages = [
    {"role": "user", "content": "Where is it located?"}
]

chat_completion = client.chat.completions.create(
    messages=messages,
    model=model
)

print(chat_completion.choices[0].message.content)

## 3. Conversations: what happens behind every chat interface

As we see, without the first message *Explain with two sentences what is the LUMI supercomputer?* , the model does not know what "Where is it located?" refers to.

**The model does not remember earlier requests.** Every request is handled separately, as if the conversation had just started.

When you use a chat interface, such as the Aitta web frontend, it looks like the model remembers what you said earlier. In reality, the chat application keeps the conversation history for you. Every time you send a new message, it sends the **whole conversation** to the model: the earlier questions, the model's earlier replies (with the role `assistant`) and your new message.

When you use the API from code, there is no chat application doing this for you, so you build the conversation history yourself.

Let's try this in practice. The function below does the same as a chat interface does behind the scenes. Every time you call `chat()`, it:

1. adds your message to the conversation history with the role `user`,
2. sends the **whole** history to the model, and
3. adds the model's reply to the history with the role `assistant`, so that it is included in the next request.

In [ ]:
# The conversation history. The system message is optional.
conversation = [
    {"role": "system", "content": "You are a helpful assistant who gives short answers."}
]

def chat(user_message):
    # 1. Add your message to the history
    conversation.append({"role": "user", "content": user_message})

    # 2. Send the whole history to the model
    response = client.chat.completions.create(
        model=model,
        messages=conversation
    )
    reply = response.choices[0].message.content

    # 3. Add the model's reply to the history
    conversation.append({"role": "assistant", "content": reply})
    return reply

Ask a first question, and then a follow-up question that only makes sense if the model knows what you talked about before:

In [58]:
print(chat("What is the LUMI supercomputer?"))

LUMI (Large Unified Modern Infrastructure) is a pre-exaflop supercomputer located in Kajaani, Finland. It's one of the world's most powerful computers, providing high-performance computing resources for scientific research and innovation in Europe.


In [59]:
print(chat("Where is it located?"))

LUMI is located in Kajaani, Finland, at the Data Center of the CSC – IT Center for Science.


In [ ]:
# Print the history: what was sent to the model with your last question
for message in conversation:
    print(f"{message['role']:>9}: {message['content']}\n")

## 4. Adjusting the parameters

Besides `model` and `messages`, you can give optional parameters that control the randomness, length and number of the generated responses. Below are some commonly used ones. See the [OpenAI API reference](https://platform.openai.com/docs/api-reference/chat/create) for all parameters, but note that Aitta does not necessarily support every feature.

| Parameter | Description | Effect |
|-----------|-------------|--------|
| **temperature** (randomness) | A value between 0 and 2 that controls how random the choice of the next token is. | Higher values give more varied and creative responses, lower values more focused and predictable ones. |
| **top_p** (nucleus sampling) | A value between 0 and 1. The model chooses the next token only from the most likely options, whose probabilities add up to `top_p`. | Lower values (like 0.5) make the model more focused and predictable, higher values (like 0.9) more varied. |
| **max_completion_tokens** (maximum length) | The maximum number of tokens (words or parts of words) the model can generate in the response. | A smaller value gives shorter responses, but the response is cut off if the limit is reached. |
| **n** (number of responses) | The number of alternative responses to generate for the same input. | A higher value (e.g. 3) gives several variations to compare. |

*Tip: Typically, adjusting either `temperature` or `top_p` is enough. Setting both too high may lead to overly random responses, and setting both too low to overly constrained ones.*

**Why are responses sometimes cut off?** There are two common reasons:
* The response reaches the `max_completion_tokens` limit.
* The input and the response together exceed the model's maximum context length, which is 8192 tokens for Llama-Poro-2-70B-Instruct.

There is also the parameter `stream`, which does not change the response itself but how it is delivered: with `stream=True`, you get the response in small pieces while it is being generated, like in chat applications. Streaming is covered in the notebook.

## Try it yourself

Now it's time for you to experiment with everything you have learned in this notebook. Use the empty cells below, and copy code from the earlier cells as a starting point.

**1. Chat completions and roles**
* Ask the model a question of your own with a single `user` message.
* Uncomment a `system` message and ask the same question again. Try e.g. *"Answer like a pirate"* or *"Answer in Finnish"*. How does the response change?

In [50]:
prompt = "< Add you question here>"

In [ ]:
chat_completion = client.chat.completions.create(
    model=model,
    messages=[
        #{"role": "system", "content": "Answer like a pirate."},     # try e.g. "Answer in Finnish."
        {"role": "user", "content": prompt}     
    ]
)

print(chat_completion.choices[0].message.content)

**2. Parameters**
* Ask the same creative question (e.g. *"Suggest a name for a reindeer café"*) with `temperature=0.1` and `temperature=1.5`. Run each a few times. Which one gives more varied answers?
* Set `max_completion_tokens=20` and ask for a longer explanation. What happens to the response?
* Use `n=3` to get three alternative responses at once, and print them all.

*Tip:* With `n=3`, the responses are in `chat_completion.choices`, so you can print them with a loop:
`for choice in chat_completion.choices: print(choice.message.content)`

In [ ]:
temperature=            # randomness: 0 = predictable, higher = more varied
max_completion_tokens=   # maximum length of each response in tokens
n=                          # number of alternative responses

In [ ]:
# A starting point: change the messages and parameters and run the cell again
chat_completion = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},        # sets the model's behaviour
        {"role": "user", "content": "Suggest a good name for a reindeer café."} # your question
    ],
    temperature=temperature, 
    max_completion_tokens=max_completion_tokens,
    n=n
)

# chat_completion.choices is a list with one response per n
for choice in chat_completion.choices:
    print(choice.message.content)   # the generated text of this response
    print("-" * 40)